# Compare the demo outputs with COSMOS2020 LePhare

Run `broadband_flow_demo.ipynb`, `bootstrapped_flow_demo.ipynb`, and
`noisesom_demo.ipynb` first. This notebook **does not train models or download data**.
It consumes their `predictions.npz` and `run.json`, and `data/parent.npz` for the
released full-multiband LePhare PDFs. Use the project's Conda kernel.

The primary paper figure is **NMAD versus magnitude**, using the paper's bins,
actual bin-median magnitudes, galaxy counts and 200 paired galaxy-bootstrap
resamples. Accuracy and coverage plots are also produced. New PDF names end in
`_demo_comparison`; the manuscript's frozen figure inputs and numeric macros are
not overwritten. Executed copies belong in `results/notebooks_executed/`.

Interpretation matters: the flows use fresh noise augmentation during training;
the NoiseSOM demo uses fixed degraded training measurements. The bootstrapped
flow uses more training objects and probabilistic labels. LePhare uses deep
multiband information and supplies those labels. These are comparisons of the
**demo outputs as run**, not the separate matched-training NoiseSOM control or a
claim that the methods use equal information. Coverage describes the selected
spectroscopic test sample, not the full photometric population.

## 1. Select saved runs and plotting settings

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'setup.py').exists())
sys.path.insert(0, str(root / 'src'))
from pzflow.utils import PDFSampler, coverage

# Edit these paths to compare other completed demo runs; no checkpoint is needed.
parent_path = root / 'data/parent.npz'
inputs = {
    'broadband': root / 'results/broadband_flow_demo',
    'bootstrapped': root / 'results/bootstrapped_flow_demo',
    'noisesom': root / 'results/noisesom_demo',
}
out_dir = root / 'results/paper_demo_comparison'
paper_dir = root / 'tex/figures'
export_paper = True
use_tex = True  # Set False if system LaTeX is unavailable.
bootstrap_draws = 200
bootstrap_seed = 20261124
magnitude_edges = np.array([-np.inf, 22.5, 24.0, 24.5, 25.0])
models = [
    ('broadband', 'Spec-z broadband flow', 'C1'),
    ('bootstrapped', 'COSMOS-PDF flow', 'C0'),
    ('noisesom', 'NoiseSOM', 'C2'),
    ('lephare', 'LePhare (full COSMOS)', 'C3'),
]
if use_tex and not shutil.which('latex'):
    raise RuntimeError('Install system LaTeX or set use_tex=False above')
plt.rcParams.update(
    {
        'text.usetex': use_tex,
        'font.family': 'serif',
        'font.size': 13,
        'axes.linewidth': 1.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'legend.frameon': False,
    }
)
out_dir.mkdir(parents=True, exist_ok=True)
if export_paper:
    paper_dir.mkdir(parents=True, exist_ok=True)

## 2. Fix the evaluation population and verify each artifact

Use every secure-spec galaxy in parent split 2. Join by Classic ID, independently
of saved row order. Missing IDs, duplicate IDs, mismatched parent hashes or different
flux/error realizations stop execution instead of silently reducing the sample.
Only matching prediction fields are used; the full photometric-test quantile files
are intentionally not substituted for the spectroscopic evaluation artifacts.

In [ ]:
def file_hash(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()


def align_prediction(path, target_ids):
    with np.load(path) as f:
        source_ids = f['id']
        if len(np.unique(source_ids)) != len(source_ids):
            raise ValueError(f'Duplicate Classic IDs in {path}')
        order = np.argsort(source_ids)
        sorted_ids = source_ids[order]
        j = np.searchsorted(sorted_ids, target_ids)
        if np.any(j == len(sorted_ids)) or not np.array_equal(sorted_ids[j], target_ids):
            raise ValueError(f'Missing required test galaxies in {path}')
        rows = order[j]
        return {k: f[k][rows] for k in f.files if k != 'grid'}


parent_hash = file_hash(parent_path)
with np.load(parent_path) as parent:
    if not json.loads(str(parent['metadata'])).get('canonical_parent'):
        raise ValueError('Use the canonical parent from match_cosmos.py')
    parent_ids = parent['id']
    if len(np.unique(parent_ids)) != len(parent_ids):
        raise ValueError('Duplicate parent IDs')
    test = np.flatnonzero((parent['split'] == 2) & np.isfinite(parent['zspec']))
    truth = {k: parent[k][test] for k in ['id', 'zspec', 'imag', 'flux', 'fluxerr']}
    # Extract only the required rows after decompressing the parent PDF array.
    lephare_grid, lephare_pdf = parent['grid'], parent['pdf'][test]
if not len(test):
    raise ValueError('No secure-spec test galaxies')

predictions, manifests = {}, {}
prefixes = {'broadband': 'specz', 'bootstrapped': 'bootstrapped', 'noisesom': 'som'}
for key, directory in inputs.items():
    manifest = json.loads((directory / 'run.json').read_text())
    if manifest['parent_sha256'] != parent_hash:
        raise ValueError(f'{key} was produced from a different parent')
    saved = align_prediction(directory / 'predictions.npz', truth['id'])
    for field in ['zspec', 'imag', 'flux', 'fluxerr']:
        if not np.array_equal(saved[field], truth[field]):
            raise ValueError(f'{key}: mismatched {field}')
    prefix = prefixes[key]
    predictions[key] = {'quantiles': saved[prefix + '_quantiles'], 'pit': saved[prefix + '_pit']}
    manifests[key] = manifest
print(f'Using all {len(test):,} secure-spec test galaxies for every method')

## 3. Derive LePhare medians, intervals and PIT from the released PDFs

Normalize each density by trapezoidal integration. Quantiles invert the CDF of
the piecewise-linear density continuously within grid cells, using the same
quadratic inversion as the shared posterior sampler. No prior correction,
recalibration or catalog best-fit redshift replaces the released posterior.
All four point estimates are posterior medians.

In [ ]:
def pdf_quantiles(sampler, probs=(0.025, 0.16, 0.5, 0.84, 0.975)):
    rows = np.arange(len(sampler.p))
    columns = []
    for prob in probs:
        # Locate the CDF cell, then solve its integrated linear density exactly.
        j = np.clip((sampler.c <= prob).sum(axis=1) - 1, 0, len(sampler.grid) - 2)
        width = np.diff(sampler.grid)[j]
        a = sampler.p[rows, j]
        slope = (sampler.p[rows, j + 1] - a) / width
        mass = np.maximum(prob - sampler.c[rows, j], 0)
        den = a + np.sqrt(np.maximum(a * a + 2 * slope * mass, 0))
        step = np.divide(2 * mass, den, out=np.zeros_like(mass), where=den > 0)
        columns.append(sampler.grid[j] + np.clip(step, 0, width))
    return np.column_stack(columns)


sampler = PDFSampler(lephare_grid, lephare_pdf)
predictions['lephare'] = {'quantiles': pdf_quantiles(sampler), 'pit': sampler.pit(truth['zspec'])}
del sampler, lephare_pdf
for key, pred in predictions.items():
    q, pit = pred['quantiles'], pred['pit']
    assert q.shape == (len(test), 5) and pit.shape == (len(test),)
    assert np.isfinite(q).all() and np.isfinite(pit).all()
    assert np.all(np.diff(q, axis=1) >= 0) and np.all((pit >= 0) & (pit <= 1))

## 4. Compare accuracy and interval coverage

In [ ]:
def summarize(z, q, pit):
    residual = (q[:, 2] - z) / (1 + z)
    c68, c95 = coverage(pit)
    return {
        'n': len(z),
        'bias': float(np.median(residual)),
        'nmad': float(1.4826 * np.median(abs(residual - np.median(residual)))),
        'outlier_fraction': float(np.mean(abs(residual) > 0.15)),
        'coverage68': float(c68),
        'coverage95': float(c95),
        'median_width68': float(np.median(q[:, 3] - q[:, 1])),
    }


z = truth['zspec']
summary = {key: summarize(z, pred['quantiles'], pred['pit']) for key, pred in predictions.items()}
print(f'{"Method":28s} {"NMAD":>8s} {"Outliers":>10s} {"C68":>8s} {"C95":>8s}')
for key, label, _ in models:
    m = summary[key]
    print(
        f'{label:28s} {m["nmad"]:8.4f} {100 * m["outlier_fraction"]:9.2f}% '
        f'{100 * m["coverage68"]:7.1f}% {100 * m["coverage95"]:7.1f}%'
    )

## 5. Primary paper figure: magnitude-dependent scatter

Bins are i<22.5, 22.5≤i<24, 24≤i<24.5 and 24.5≤i<25, with **no extra bright cut**.
Each point is at the median magnitude of that same bin's galaxies. Use the same
200 resampled galaxy index arrays for all methods, so method differences remain
paired. Bands are pointwise 95% percentile intervals with alpha=0.5. These intervals
condition on the trained models; they exclude training variation, spatial covariance
and spectroscopic-selection uncertainty.

In [ ]:
mag = truth['imag']
rng = np.random.default_rng(bootstrap_seed)
centers, counts = [], []
binned = {key: [] for key, _, _ in models}
limits = {key: [] for key, _, _ in models}
for lo, hi in zip(magnitude_edges[:-1], magnitude_edges[1:]):
    idx = np.flatnonzero((mag >= lo) & (mag < hi))
    if not len(idx):
        raise ValueError(f'Empty magnitude bin [{lo}, {hi})')
    centers.append(float(np.median(mag[idx])))
    counts.append(len(idx))
    draws = rng.choice(idx, size=(bootstrap_draws, len(idx)), replace=True)
    for key, _, _ in models:
        q, pit = predictions[key]['quantiles'], predictions[key]['pit']
        binned[key].append(summarize(z[idx], q[idx], pit[idx]))
        scatter = [summarize(z[d], q[d], pit[d])['nmad'] for d in draws]
        limits[key].append(np.quantile(scatter, [0.025, 0.975]))
assert sum(counts) == len(test)
fig, ax = plt.subplots(figsize=(7, 4.5))
for key, label, color in models:
    value = np.array([m['nmad'] for m in binned[key]])
    band = np.array(limits[key])
    ax.plot(centers, value, 'o-', color=color, label=label)
    ax.fill_between(centers, band[:, 0], band[:, 1], color=color, alpha=0.5)
for x, count in zip(centers, counts):
    ax.text(x, 1.02, f'$N={count}$', transform=ax.get_xaxis_transform(), ha='center', fontsize=10)
ax.set(xlabel='HSC i aperture magnitude [AB]', ylabel=r'$\sigma_{\rm NMAD}$')
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(out_dir / 'magnitude.png', dpi=180)
if export_paper:
    fig.savefig(paper_dir / 'fig04_magnitude_dependence_demo_comparison.pdf')
print('Bin counts:', counts)
print('Effective magnitudes:', centers)

## 6. Supporting accuracy and calibration figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
for ax, (key, label, color) in zip(axes.flat, models):
    ax.scatter(z, predictions[key]['quantiles'][:, 2], s=3, alpha=0.2, color=color, rasterized=True)
    ax.plot([0, 6], [0, 6], 'k--', lw=1)
    ax.set(title=label, xlim=(0, 6), ylim=(0, 6))
for ax in axes[-1]:
    ax.set_xlabel(r'$z_{\rm spec}$')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$z_{\rm median}$')
fig.tight_layout()
fig.savefig(out_dir / 'accuracy.png', dpi=180)
if export_paper:
    fig.savefig(paper_dir / 'fig02_accuracy_demo_comparison.pdf')

In [ ]:
levels = np.linspace(0, 1, 101)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, label, color in models:
    pit = predictions[key]['pit']
    axes[0].plot(levels, coverage(pit, levels), label=label, color=color)
    axes[1].hist(pit, bins=20, range=(0, 1), density=True, histtype='step', color=color)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[0].set(
    xlabel='Nominal central probability', ylabel='Empirical coverage', xlim=(0, 1), ylim=(0, 1)
)
axes[1].set(xlabel='PIT', ylabel='Probability density', xlim=(0, 1))
axes[0].legend(fontsize=10)
fig.tight_layout()
fig.savefig(out_dir / 'calibration.png', dpi=180)
if export_paper:
    fig.savefig(paper_dir / 'fig03_calibration_demo_comparison.pdf')

## 7. Save compact figure inputs and provenance

The saved comparison is keyed by the common Classic IDs. It records all input
hashes, original demo manifests, bootstrap settings and the exact magnitude bins.
The released LePhare posterior remains the reference without prior correction.
No manuscript text, frozen figure, model, or original prediction file is modified.

In [ ]:
arrays = dict(truth)
for key, pred in predictions.items():
    arrays[key + '_quantiles'] = pred['quantiles']
    arrays[key + '_pit'] = pred['pit']
np.savez_compressed(out_dir / 'predictions.npz', **arrays)
report = {
    'parent_sha256': parent_hash,
    'n_common_test': len(test),
    'metrics': summary,
    'magnitude_edges': [None, *magnitude_edges[1:].tolist()],
    'lower_edge_none_means': '-infinity; no bright cut',
    'bin_counts': counts,
    'effective_magnitudes': centers,
    'bin_metrics': binned,
    'nmad_95_intervals': {key: np.array(band).tolist() for key, band in limits.items()},
    'bootstrap': {
        'draws': bootstrap_draws,
        'seed': bootstrap_seed,
        'method': 'paired within-bin galaxy bootstrap; trained models held fixed',
    },
    'input_files': {
        key: {'path': str(path), 'sha256': file_hash(path)}
        for key, path in [
            ('parent', parent_path),
            *[
                (name + '_predictions', directory / 'predictions.npz')
                for name, directory in inputs.items()
            ],
            *[(name + '_run', directory / 'run.json') for name, directory in inputs.items()],
        ]
    },
    'demo_manifests': manifests,
    'numpy_version': np.__version__,
    'notes': [
        'Flows augment training noise; NoiseSOM uses fixed degraded training fluxes.',
        'LePhare uses full deep multiband information and supplies bootstrap labels.',
        'Single saved run per method; test spectroscopy is selected.',
        'These replace neither the frozen manuscript results nor the matched NoiseSOM control.',
    ],
}
with (out_dir / 'comparison.json').open('w') as f:
    json.dump(report, f, indent=2, allow_nan=False)
    f.write(chr(10))
print(f'Figures and numerical comparison saved to {out_dir}')